In [1]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import glob, os
import SFI
import SFI.OLI_bases
import jax.numpy as jnp
from jax import random
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import pandas as pd
import pickle


In [ ]:
def get_experimentID_fightbouts(path):

    tracking_folder = os.path.dirname(path)

    loadpaths = glob.glob(os.path.join(tracking_folder, "*results.h5"))
    loadpaths.sort()

    expNames = [os.path.basename(p)[:23] for p in loadpaths]

    target_expName = os.path.basename(path)[:23]
    expIdx = expNames.index(target_expName)

    fightbout_path = os.path.join(tracking_folder, "fightBouts.h5")

    with h5py.File(fightbout_path, "r") as j:
        fb = j["fight_bout_info_noDurThresh"][:]

    fightbouts = fb[fb[:, 0].astype(int) == expIdx]

    return expIdx, fightbouts
EXP_id , fightbout = get_experimentID_fightbouts("/Users/marindevandijk/Documents/CLS 2025/Thesis/Coding/ZebraFish_project/Data/tracking_results/FishTank20200130_153857_tracking_results.h5")
print(fightbout)

In [2]:
def prepare_data(path,infight):
    "Prepare the data make it ready to calculate dpp,theta1 and theta2"
    "if infight = True return data with only the infight bouts otherwise it returns total trajectory  "

    path = path
    f = h5py.File(path, "r")

    X = f["tracks_3D_smooth"][:]
    EXP_id , fightbout = get_experimentID_fightbouts(path)
    if infight == True and fightbout.size > 0:
        X_coordinates = X[fightbout[0,1]:fightbout[0,2],:,:,:]
    else:
        X_coordinates = X.copy()
    return X_coordinates,fightbout

def wrap_pi(a):
    return (a + np.pi) % (2*np.pi) - np.pi

def calculate_theta(fish0,fish1):
    vector_fish0 = (fish0[:,0,:] - fish0[:,1,:]) # difference in heading of head and pec
    orientation_fish0 = vector_fish0/np.linalg.norm(vector_fish0, axis=1, keepdims=True)

    theta0 = np.arctan2(orientation_fish0[:, 1],  orientation_fish0[:, 0])

    r_01 = fish1[:,1,:] - fish0[:,1,:] 
    phi_01 = np.arctan2(r_01[:,1],r_01[:,0]) #dy,dx

    theta0 = np.arctan2(vector_fish0[:, 1], vector_fish0[:, 0])
    psi_01 = phi_01 - theta0
    psi_01 = wrap_pi(psi_01) # wrap interval from -pi to pi
    return psi_01

In [3]:
def calculate_variables(coordinates_cleaned):
    "input are the coordinates and returns dpp,theta_i and theta_j"

    fish_i = coordinates_cleaned[:,0,:,:]
    fish_j = coordinates_cleaned[:,1,:,:]

    pec_fish_i = coordinates_cleaned[:,0,1,:]
    pec_fish_j = coordinates_cleaned[:,1,1,:]

    d_pp = np.linalg.norm((pec_fish_j-pec_fish_i), axis=1) 
    
    theta_i = calculate_theta(fish_i,fish_j)
    theta_j = calculate_theta(fish_j,fish_i)

    return d_pp, theta_i,theta_j
    
def clean_data(d_pp,theta_i,theta_j):
    mask = (np.isfinite(d_pp) &np.isfinite(theta_i) & np.isfinite(theta_j))
    return d_pp[mask],theta_i[mask],theta_j[mask]


def Build_segmented_data(dpp,theta1,theta2):
    Valid = (np.isfinite(dpp) &np.isfinite(theta1) & np.isfinite(theta2))
    valid_idx = np.where(Valid)[0]

    X_list = []
    time_list = []
    segid_list = []
    seg_ranges = []
    start = 0
    t_offset = 0
    seg_id = 0

    for k in range(1, len(valid_idx) + 1):
        if k == len(valid_idx) or valid_idx[k] != valid_idx[k - 1] + 1: # if there is the end of trajectory or a hole 

            seg_idx = valid_idx[start:k]

            if len(seg_idx) > 5:
                D_seg = dpp[seg_idx]
                th1_seg = np.unwrap(theta1[seg_idx])
                th2_seg = np.unwrap(theta2[seg_idx])

                X_list.append(np.column_stack([D_seg, th1_seg, th2_seg]))
                time_list.append(np.arange(len(seg_idx)) + t_offset)
                segid_list.append(np.full(len(seg_idx), seg_id, dtype=int))
                seg_ranges.append((seg_idx[0], seg_idx[-1]))

                t_offset += len(seg_idx) + 1
                seg_id += 1

            start = k

    X = np.vstack(X_list)
    time_idx = np.concatenate(time_list)
    segment_ids = np.concatenate(segid_list)
    return X, time_idx, segment_ids, seg_ranges

def subsample_random_segments(X, segment_ids, fraction=0.85):
    np.random.seed(5)
    unique_segments = np.unique(segment_ids)
    n_keep = int(fraction * len(unique_segments))
    keep_seg = np.random.choice(unique_segments,size=n_keep,replace= False)

    X_list = []
    time_list = []
    t_offset = 0
    for seg in keep_seg:
        indices = np.where(segment_ids == seg)[0]
        X_seg = X[indices]
        X_list.append(X_seg)
        time_list.append(np.arange(len(X_seg)) + t_offset)
        t_offset += len(X_seg) + 1

    X_new = np.vstack(X_list)
    time_idx_new = np.concatenate(time_list)
    return X_new, time_idx_new

def Run_Force_inference(X,time_idx,K,M):
    traj = SFI.StochasticTrajectoryData(X, time_idx, 0.01)
    poly_1d,poly_describe = SFI.OLI_bases.polynomial_basis(dim=1,order=K)
    fourier1d_F1 = SFI.OLI_bases.Fourier_basis(dim =1,order=M,center= jnp.array([0.0]),width = jnp.array([2*jnp.pi]))
    fourier1d_F2 = SFI.OLI_bases.Fourier_basis(dim =1,order=M,center = jnp.array([0.0]),width = jnp.array([2*jnp.pi]))
    
    def radial_basis(D):
        p_poly = poly_1d(jnp.array([D]))

        lam = jnp.array([1.0,2.5, 6.0])
        p_exp = jnp.exp(-D /lam)

        return p_exp #jnp.concatenate([p_poly,p_exp])

    def C_func(x):
        D  = x[0]
        th1 = x[1]
        th2 = x[2]

        p  = poly_1d(jnp.array([D]))
        p = radial_basis(D)      
        f1 = fourier1d_F1(jnp.array([th1]))
        f2 = fourier1d_F2(jnp.array([th2]))  
        triple = jnp.einsum('i,j,k->ijk', p, f1[1:], f2[1:]).reshape(-1)
        
        phi = jnp.concatenate([
        p, f1, f2, jnp.outer(p, f1[1:]).reshape(-1),jnp.outer(f2[1:], f1[1:]).reshape(-1),
        jnp.outer(p, f2[1:]).reshape(-1),
        triple], axis=0)
        return phi 
    
    S = SFI.OverdampedLangevinInference(traj)
    S.compute_diffusion_constant(method="MSD")
    (funcs_and_grad, descriptor) = SFI.OLI_bases.basis_selector(
        {"type": "custom_scalar", "functions": C_func},
        dimension=3,
        output="vector"
    )
  
    basis_F, grad_F = funcs_and_grad
    S.infer_force_linear(basis_linear=basis_F, basis_linear_gradient=grad_F)
    #S.sparsify_force()
    S.compute_force_error() 
    S.print_report()
    return S, descriptor

In [ ]:
path = "/Users/marindevandijk/Documents/CLS 2025/Thesis/Coding/ZebraFish_project/Data/tracking_results/FishTank20200130_153857_tracking_results.h5"
X_coordinates, fightbout = prepare_data(path,False)
print(X_coordinates.shape)

print(fightbout)
print(fightbout)
print("number of bouts:", len(fightbout))

start = int(fightbout[0, 2])
stop  = int(fightbout[1, 1])
duration = fightbout[0, 3]
d_pp_after,theta_i_after,theta_j_after = calculate_variables(X_coordinates[start:stop,:,:,:])
plt.hist(theta_i_after,density=True,alpha = 0.5,bins = 100,label = 'theta1')
plt.hist(theta_j_after,density=True,alpha = 0.5,bins = 100,label  ='theta2')
plt.title('Distribution after fight')
plt.legend()

In [ ]:
N = len(d_pp)
d_pp_clean, theta_i_clean,theta_j_clean = clean_data(d_pp,theta_i,theta_j)

d_pp_half = d_pp_clean[:N//2]
theta_i_half = theta_i_clean[:N//2]
theta_j_half = theta_j_clean[:N//2]

fig, axs = plt.subplots(1, 3, figsize=(15,4))

axs[0].hist(d_pp_half, bins=100)
axs[0].set_title(r'$d_{pp}$ Real')

axs[1].hist(theta_i_half, bins=100)
axs[1].set_title(r"$\theta_1$ Real")

axs[2].hist(theta_j_half, bins=100)
axs[2].set_title(r"$\theta_2$ Real")

plt.tight_layout()
save_path = "Exploratory_Figures"
os.makedirs(save_path, exist_ok=True)

#plt.savefig(f"{save_path}/Distributions_Real.png", dpi=300)
plt.show()

In [ ]:
def F(D_vals,theta1_vals,theta2_vals):
    states = jnp.column_stack([D_vals,theta1_vals, theta2_vals])
    F_vals = S.force_ansatz(states)
    dD_dt = np.array(F_vals[:, 0])
    dth1_dt = np.array(F_vals[:, 1])
    dth2_dt = np.array(F_vals[:, 2])
    return dD_dt,dth1_dt,dth2_dt

D_median =5#np.median(d_pp_clean)
theta1 =np.pi/2
theta2 = np.pi/2
distance_vals = np.linspace(0,np.max(d_pp_clean),400)
theta1_vals = np.linspace(-np.pi,np.pi,400)

d_dt,th1_dt,th2_dt = F(distance_vals,np.full(400, theta1),np.full(400, theta2))
os.makedirs("Exploratory_Figures", exist_ok=True)
plt.plot(distance_vals, d_dt)
plt.axhline(0, linestyle='--', color ='k')
plt.xlabel(f'd')
plt.ylabel(r'$\dot{d}_{pp}$')
plt.title(f'Distance force at theta1={theta1:.2f}, theta2={theta2:.2f}')
#plt.savefig("Exploratory_Figures/Forces/Dforce_theta1_0.5pi_theta1_-0.5pi.png")
plt.show()

In [ ]:
D_value =2.5
theta1 =np.pi/2
theta2 = np.pi/2
distance_vals = np.linspace(0,np.max(d_pp_clean),400)
theta_vals = np.linspace(-np.pi,np.pi,400)

d_dt,th1_dt,th2_dt = F(np.full(400, D_value),theta_vals,np.full(400, theta2))
os.makedirs("Exploratory_Figures", exist_ok=True)
plt.plot(theta_vals,th1_dt)
plt.axhline(0, linestyle='--', color ='k')
plt.xlabel(r'$\theta_1$')
plt.ylabel(r'$\dot{\theta}_{1}$')
plt.title(f' Angle 1 force at Distance={D_value:.2f}, theta2={theta2:.2f}')
plt.savefig("Exploratory_Figures/Forces/th1Force_Distance_2.5_th2_0.5pi.png")
plt.show()

In [ ]:
D_value =2.5#np.median(d_pp_clean)
theta1 =np.pi/2
theta2 = np.pi/2
distance_vals = np.linspace(0,np.max(d_pp_clean),400)
theta_vals = np.linspace(-np.pi,np.pi,400)

d_dt,th1_dt,th2_dt = F(np.full(400, D_value),np.full(400, theta1),theta_vals)
os.makedirs("Exploratory_Figures", exist_ok=True)
plt.plot(theta_vals,th2_dt)
plt.axhline(0, linestyle='--', color ='k')
plt.xlabel(r'$\theta_2$')
plt.ylabel(r'$\dot{\theta}_{2}$')
plt.title(f' Angle 2 force at Distance={D_value:.2f}, theta1={theta1:.2f}')
#plt.savefig("Exploratory_Figures/Forces/th2Force_Distance_2.5_th1_0.5pi.png")
plt.show()

In [ ]:
D_value = 10
theta_vals = np.linspace(-np.pi,np.pi,200)
th1, th2 = np.meshgrid(theta_vals,theta_vals)

d_dt,th1_dt,th2_dt = F(np.full(th1.size, D_value),th1.ravel(), th2.ravel())

d_dt = d_dt.reshape(th1.shape)
v = np.max(abs(d_dt))

plt.imshow(d_dt,cmap = 'RdBu',extent=[-np.pi, np.pi, -np.pi, np.pi],vmin=-v,vmax=v,
    origin='lower')
plt.colorbar(label=r'$F_D$')
plt.xlabel(r'$\theta_1$')
plt.ylabel(r'$\theta_2$')
plt.title(r'Heatmap force $F_D$')

In [ ]:
D_value = 6
theta_vals = np.linspace(-np.pi,np.pi,200)
th1, th2 = np.meshgrid(theta_vals,theta_vals)

d_dt,th1_dt,th2_dt = F(np.full(th1.size, D_value),th1.ravel(), th2.ravel())

th2_dt = th2_dt.reshape(th1.shape)
v = np.max(np.abs(th2_dt))
plt.imshow(th2_dt,cmap = 'RdBu',extent=[-np.pi, np.pi, -np.pi, np.pi],vmin=-v,vmax=v,
    origin='lower')
plt.colorbar(label=r'$F_{\theta_2}$')
plt.xlabel(r'$\theta_1$')
plt.ylabel(r'$\theta_2$')
plt.title(r'Heatmap force $\theta_2$')
plt.savefig("Exploratory_Figures/Forces/heatmap_th2Force_Distance_6.png")

In [ ]:
D_value = 6
theta_vals = np.linspace(-np.pi,np.pi,200)
th1, th2 = np.meshgrid(theta_vals,theta_vals)

d_dt,th1_dt,th2_dt = F(np.full(th1.size, D_value),th1.ravel(), th2.ravel())

th1_dt = th1_dt.reshape(th1.shape)
v = np.max(np.abs(th1_dt))
plt.imshow(th1_dt,cmap = 'RdBu',extent=[-np.pi, np.pi, -np.pi, np.pi],vmin=-v,vmax=v,
    origin='lower')
plt.colorbar(label=r'$F_{\theta_1}$')
plt.xlabel(r'$\theta_1$')
plt.ylabel(r'$\theta_2$')
plt.title(r'Heatmap force $\theta_1$')
plt.savefig("Exploratory_Figures/Forces/heatmap_th1Force_Distance_6.png")

In [ ]:
t0 = np.random.randint(0, len(d_pp_clean))
x0 = [d_pp_clean[t0], theta_i_clean[t0], theta_j_clean[t0]]
traj_sim = Simulation_Dfixed(S, x0, dt=0.01, N_steps=5000,D=2.5,early_stop= True,force_tol = 1e-4,n_consecutive = 20)

theta1 = traj_sim[:,1]
theta2 = traj_sim[:,2]
print(traj_sim[-1,0])
plt.plot(theta1,theta2)
plt.xlabel(r'$\theta_1$')
plt.ylabel(r'$\theta_2$')

In [ ]:
dt = 0.01
for i in range(20):
    x0 =  [ np.random.uniform(0,10), -1.05999994,-0.125 ]
    traj_sim_dpp = Simulation_anglesfixed(S, x0[0], x0[1], x0[2], dt=0.01, N_steps=5000,
                      early_stop=True, force_tol=1e-4, n_consecutive=20)
    t = np.arange(len(traj_sim_dpp)) * dt
    plt.plot(t,traj_sim_dpp[:len(t),0])

plt.xlabel('Time')
plt.ylabel(f'D')
#plt.text(t[380], D_final +0.2, f' D final = {D_final:.2f}', color='red')
plt.title(f' D over time, when th1={x0[1]:.2f}, th2={x0[2]:.2f}')
"""
x0 = [ 3 , 2.62800002 , 0.155]
x1 = [ 2 , 2.62800002 , 0.155]
x2 = [ 8 , 2.62800002 , 0.155]
x3 = [ 6 , 2.62800002 , 0.155]
x4 = [ 1 , 2.62800002 , 0.155]
traj_sim_dpp = Simulation_deterministic(S, x0, dt=0.01, N_steps=5000,D=None,theta1 =x0[1], theta2 = x0[2],early_stop= True,force_tol = 1e-4,n_consecutive = 20)
traj_sim_dpp_1 = Simulation_deterministic(S, x1, dt=0.01, N_steps=5000,D=None,theta1 =x1[1], theta2 = x1[2],early_stop= True,force_tol = 1e-4,n_consecutive = 20)
traj_sim_dpp_2 = Simulation_deterministic(S, x2, dt=0.01, N_steps=5000,D=None,theta1 =x2[1], theta2 = x2[2],early_stop= True,force_tol = 1e-4,n_consecutive = 20)
traj_sim_dpp_3 = Simulation_deterministic(S, x3, dt=0.01, N_steps=5000,D=None,theta1 =x3[1], theta2 = x3[2],early_stop= True,force_tol = 1e-4,n_consecutive = 20)
traj_sim_dpp_4 = Simulation_deterministic(S, x4, dt=0.01, N_steps=5000,D=None,theta1 =x4[1], theta2 = x4[2],early_stop= True,force_tol = 1e-4,n_consecutive = 20)


D_final = traj_sim_dpp[-1,0]
plt.plot(t,traj_sim_dpp[:,0])
plt.plot(t,traj_sim_dpp_1[:,0])
plt.plot(t,traj_sim_dpp_2[:,0])
plt.plot(t,traj_sim_dpp_3[:,0])
plt.plot(t,traj_sim_dpp_4[:,0])

"""